In [ ]:
import re
import time
import random
import json
from urllib.parse import urljoin

import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

In [ ]:
# =========================
# 설정
# =========================
TARGET_ACCOUNT       = "rextreme.official"
OUTPUT_FILE          = "rextreme_all_posts_full_v5.csv"
CHROME_MAJOR_VERSION = 145

MAX_EXPAND_CLICKS = 30
MAX_COMMENTS      = 100

# 스크롤 종료 조건
# scrollHeight 변화 없는 횟수가 NO_CHANGE_LIMIT 이상이면 끝으로 판단
NO_CHANGE_LIMIT = 5

In [ ]:
# =========================
# 대기 유틸
# =========================
def pause(a=1.0, b=2.0):
    time.sleep(random.uniform(a, b))


def long_pause(prob=0.10, a=4.0, b=8.0):
    if random.random() < prob:
        t = random.uniform(a, b)
        print(f"long sleep: {round(t, 2)} sec")
        time.sleep(t)


In [ ]:
# =========================
# 드라이버 생성
# =========================
def create_driver():
    options = uc.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = uc.Chrome(
        options=options,
        version_main=CHROME_MAJOR_VERSION,
        use_subprocess=True,
    )
    driver.implicitly_wait(5)
    return driver

In [ ]:
# =========================
# 텍스트 / 숫자 유틸
# =========================
def safe_text(elem):
    try:
        txt = elem.get_attribute("textContent")
        return txt.strip() if txt else ""
    except:
        return ""


def extract_number(text):
    if not text:
        return None
    nums = re.findall(r"[\d,]+", text)
    if not nums:
        return None
    try:
        return int(nums[0].replace(",", ""))
    except:
        return None


def extract_hashtags(text):
    if not text:
        return []
    return re.findall(r"#[^\s#@]+", text)


def extract_mentions(text):
    if not text:
        return []
    return re.findall(r"@[A-Za-z0-9._]+", text)


In [ ]:
# =========================
# 요소 대기
# =========================
def wait_until_any(driver, selectors, timeout=25):
    end = time.time() + timeout
    while time.time() < end:
        for by, sel in selectors:
            try:
                elems = driver.find_elements(by, sel)
                if elems:
                    return True
            except:
                pass
        time.sleep(0.7)
    return False

In [ ]:
# =========================
# 프로필 총 게시물 수 감지
# =========================
def get_profile_post_count(driver):
    candidates = []
    xpaths = [
        "//header//li",
        "//main//header//li",
        "//meta[@property='og:description']",
        "//meta[@name='description']",
    ]
    for xp in xpaths:
        try:
            elems = driver.find_elements(By.XPATH, xp)
            for e in elems:
                txt = e.get_attribute("content") if e.tag_name == "meta" else safe_text(e)
                if txt:
                    candidates.append(txt)
        except:
            pass

    for txt in candidates:
        m = re.search(r"게시물\s*([\d,]+)", txt)
        if m:
            return int(m.group(1).replace(",", ""))
        m = re.search(r"([\d,]+)\s+posts", txt, flags=re.I)
        if m:
            return int(m.group(1).replace(",", ""))
    return None


In [ ]:
# =========================
# 게시물 링크 전체 수집
#
# [원인 분석]
# window.scrollTo(0, body.scrollHeight) 방식이 프로필 페이지에서 실패하는 이유:
#   - 프로필 그리드가 내부 컨테이너에 렌더링 → body.scrollHeight 고정
#   - scrollTo(0, 고정값) → 매번 같은 위치 → scroll 이벤트 미발생
#   - Instagram 로딩 트리거 안됨 → 12개에서 멈춤
#
# [해결]
# window.scrollBy(0, step) 방식으로 변경
#   - scrollY(현재 위치) 기준으로 매번 새로운 위치로 이동
#   - body.scrollHeight 에 의존하지 않음
#   - scrollY 가 더 이상 증가하지 않으면 → 실제 끝 도달로 판단
#   - step = window.innerHeight * 0.8 → 화면 단위 점진 이동
# =========================
def collect_all_post_links(driver, account):
    profile_url = f"https://www.instagram.com/{account}/"
    driver.get(profile_url)
    pause(3, 5)

    ok = wait_until_any(
        driver,
        [
            (By.TAG_NAME, "main"),
            (By.XPATH, "//a[contains(@href, '/p/')]"),
            (By.XPATH, "//a[contains(@href, '/reel/')]"),
        ],
        timeout=25,
    )
    if not ok:
        raise RuntimeError("프로필 페이지 로딩 실패")

    target_count = get_profile_post_count(driver)
    print(f"프로필 총 게시물 수(감지): {target_count}")

    collected       = set()
    base_url        = "https://www.instagram.com"
    no_move_count   = 0   # scrollY 변화 없는 횟수
    scroll_count    = 0

    # 초기 로딩 충분히 대기
    pause(3.0, 2.0)

    while True:
        # ── 현재 DOM 링크 수집 ───────────────────────────
        hrefs = driver.execute_script("""
            return Array.from(document.querySelectorAll('a[href]'))
                .map(a => a.getAttribute('href'))
                .filter(Boolean);
        """)
        added = 0
        for href in hrefs:
            full = urljoin(base_url, href).split("?")[0]
            if "/p/" in full or "/reel/" in full:
                if full not in collected:
                    collected.add(full)
                    added += 1

        # 스크롤 전 현재 scrollY 기록
        scroll_y_before = driver.execute_script("return window.scrollY;")

        scroll_count += 1
        print(
            f"[scroll {scroll_count}] "
            f"collected={len(collected)} | "
            f"added={added} | "
            f"scrollY={scroll_y_before} | "
            f"no_move={no_move_count}"
        )

        # 목표 달성 시 종료
        if target_count and len(collected) >= target_count:
            print("목표 게시물 수 도달")
            break

        # ── scrollBy 로 점진적 이동 ─────────────────────
        # body.scrollHeight 에 의존하지 않고 현재 위치 기준으로 이동
        driver.execute_script("""
            window.scrollBy(0, Math.floor(window.innerHeight * 0.8));
        """)

        # Instagram 로딩 대기
        pause(3.0, 2.0)

        # scrollY 변화 감지 (실제 이동 여부 확인)
        scroll_y_after = driver.execute_script("return window.scrollY;")

        if scroll_y_after <= scroll_y_before:
            # scrollY 가 증가하지 않음 → 실제 페이지 끝 도달
            no_move_count += 1
            print(f"  scrollY 변화 없음 ({no_move_count}/{NO_CHANGE_LIMIT})")
            if no_move_count >= NO_CHANGE_LIMIT:
                print("  스크롤 끝 도달. 링크 수집 완료.")
                break
        else:
            # 이동 성공 → 카운터 리셋
            no_move_count = 0

        long_pause()

    links = sorted(collected)
    if not links:
        raise RuntimeError("게시물 링크를 하나도 수집하지 못함")

    print(f"\n최종 수집 링크 수: {len(links)}")
    return links


In [ ]:
# =========================
# 게시물 메타 정보 수집
# =========================
def get_meta_description(driver):
    xpaths = [
        "//meta[@property='og:description']",
        "//meta[@name='description']",
    ]
    for xp in xpaths:
        try:
            elems = driver.find_elements(By.XPATH, xp)
            for e in elems:
                content = e.get_attribute("content")
                if content:
                    return content.strip()
        except:
            pass
    return None


def parse_meta_description(meta_text):
    result = {
        "caption":       None,
        "like_text":     None,
        "like_count":    None,
        "comment_text":  None,
        "comment_count": None,
    }
    if not meta_text:
        return result

    korean_like     = re.search(r"(좋아요\s*[\d,]+개)", meta_text)
    korean_comment  = re.search(r"(댓글\s*[\d,]+개)", meta_text)
    english_like    = re.search(r"([\d,]+\s+likes?)", meta_text, flags=re.I)
    english_comment = re.search(r"([\d,]+\s+comments?)", meta_text, flags=re.I)

    if korean_like:
        result["like_text"]  = korean_like.group(1)
        result["like_count"] = extract_number(korean_like.group(1))
    elif english_like:
        result["like_text"]  = english_like.group(1)
        result["like_count"] = extract_number(english_like.group(1))

    if korean_comment:
        result["comment_text"]  = korean_comment.group(1)
        result["comment_count"] = extract_number(korean_comment.group(1))
    elif english_comment:
        result["comment_text"]  = english_comment.group(1)
        result["comment_count"] = extract_number(english_comment.group(1))

    quoted = re.findall(r'[""](.*?)[""]', meta_text, flags=re.S)
    if quoted:
        result["caption"] = quoted[-1].strip()

    return result

In [ ]:
def expand_comments(driver, max_clicks=30):
    total = 0

    try:
        container = driver.find_element(
            By.XPATH,
            "//div[contains(@class,'x5yr21d') and contains(@class,'xw2csxc') and contains(@class,'x1odjw0f')]"
        )
    except:
        container = None

    if not container:
        return total

    prev_count = 0
    no_change = 0

    for _ in range(max_clicks):
        # 답글 버튼 전부 클릭
        try:
            buttons = driver.find_elements(
                By.XPATH,
                "//*[@role='button'][contains(., '답글') and contains(., '보기')]"
            )
            for b in buttons:
                try:
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                    pause(0.3, 0.6)
                    driver.execute_script("arguments[0].click();", b)
                    total += 1
                    pause(0.8, 1.2)
                except:
                    pass
        except:
            pass

        # 스크롤
        spans = driver.find_elements(By.XPATH, "//span[@dir='auto']")
        curr_count = len(spans)

        if curr_count == prev_count:
            no_change += 1
            if no_change >= 3:
                break
        else:
            no_change = 0
            total += 1

        prev_count = curr_count
        driver.execute_script("arguments[0].scrollTop += 800;", container)
        pause(1.5, 2.5)

    return total

In [ ]:
def extract_dom_comments(driver, max_comments=100):
    comments = []
    seen = set()

    date_pattern = re.compile(r'^\d+[주일시분개월년]')

    bad_texts = {
        "좋아요", "답글 달기", "번역 보기", "게시물 작성자",
        "Reply", "See translation", "View replies", "View more replies",
        "팔로우", "Follow", "Following", "댓글을 남겨보세요.", "모두 보기",
        "홈", "릴스", "검색", "만들기", "알림", "프로필", "더 보기",
        "Meta", "소개", "블로그", "채용 정보", "도움말", "API",
        "개인정보처리방침", "약관", "위치", "Instagram Lite", "Threads",
        "연락처 업로드 & 비사용자", "한국어",
    }

    try:
        spans = driver.find_elements(By.XPATH, "//span[@dir='auto']")
        texts = [s.get_attribute("textContent").strip() for s in spans]
    except:
        return comments

    # 댓글 영역 시작: 캡션 다음 첫 번째 날짜 패턴 이후부터
    # 댓글 영역 끝: "Meta" 또는 "© 20" 등 푸터 시작 전까지
    start_idx = None
    end_idx = len(texts)

    for i, t in enumerate(texts):
        # 캡션 이후 첫 날짜 = 댓글 시작 직전
        if start_idx is None and i > 3 and date_pattern.match(t):
            start_idx = i - 1  # 날짜 바로 앞이 유저명
            break

    for i, t in enumerate(texts):
        if t == "Meta" or t.startswith("© 20"):
            end_idx = i
            break

    if start_idx is None:
        return comments

    texts = texts[start_idx:end_idx]

    i = 0
    while i < len(texts) - 1:
        username = texts[i]

        # 유저명 조건
        if (
            not username
            or username in bad_texts
            or len(username) > 40
            or " " in username
            or date_pattern.match(username)
        ):
            i += 1
            continue

        # 인증됨 제거
        username = username.replace("인증됨", "").strip()

        j = i + 1

        # 중복 유저명 스킵
        if j < len(texts) and (texts[j] == username or username in texts[j]):
            j += 1

        # 날짜 스킵
        if j < len(texts) and date_pattern.match(texts[j]):
            j += 1

        if j >= len(texts):
            i += 1
            continue

        comment_text = texts[j]

        if comment_text in bad_texts or len(comment_text) < 2:
            i = j + 1
            continue

        sig = (username, comment_text)
        if sig not in seen:
            seen.add(sig)
            comments.append({
                "username": username,
                "comment_text": comment_text,
                "hashtags": extract_hashtags(comment_text),
                "mentions": extract_mentions(comment_text),
            })
            if len(comments) >= max_comments:
                break

        i = j + 1

    return comments

In [ ]:
# =========================
# 게시 일시 수집
# =========================
def extract_datetime(driver):
    try:
        return driver.find_element(By.TAG_NAME, "time").get_attribute("datetime")
    except:
        return None

In [ ]:
# =========================
# 게시물 1개 크롤링
# =========================
def crawl_one_post(driver, post_url):
    driver.get(post_url)
    pause(2.0, 3.5)

    ok = wait_until_any(
        driver,
        [
            (By.TAG_NAME, "article"),
            (By.XPATH, "//meta[@property='og:description']"),
            (By.XPATH, "//meta[@name='description']"),
            (By.TAG_NAME, "time"),
        ],
        timeout=25,
    )

    empty = {
        "post_url":         post_url,
        "datetime":         None,
        "caption":          None,
        "hashtags":         "",
        "mentions":         "",
        "like_text":        None,
        "like_count":       None,
        "comment_text":     None,
        "comment_count":    None,
        "comments_json":    "[]",
        "comments_texts":   "",
        "share_count":      None,
        "expanded_clicks":  0,
        "meta_description": None,
    }

    if not ok:
        return {**empty, "status": "load_fail"}

    meta_text       = get_meta_description(driver)
    parsed          = parse_meta_description(meta_text)
    expanded_clicks = expand_comments(driver, max_clicks=MAX_EXPAND_CLICKS)
    pause(2.0, 3.0)
    comments        = extract_dom_comments(driver, max_comments=MAX_COMMENTS)

    caption        = parsed["caption"]
    comments_texts = " || ".join([c["comment_text"] for c in comments])
    print(f"| comments_found: {len(comments)}개")

    return {
        "post_url":         post_url,
        "datetime":         extract_datetime(driver),
        "caption":          caption,
        "hashtags":         " || ".join(extract_hashtags(caption)),
        "mentions":         " || ".join(extract_mentions(caption)),
        "like_text":        parsed["like_text"],
        "like_count":       parsed["like_count"],
        "comment_text":     parsed["comment_text"],
        "comment_count":    parsed["comment_count"],
        "comments_json":    json.dumps(comments, ensure_ascii=False),
        "comments_texts":   comments_texts,
        "share_count":      None,
        "expanded_clicks":  expanded_clicks,
        "meta_description": meta_text,
        "status":           "ok",
    }


In [ ]:
# =========================
# 메인
# =========================
def main():
    driver = create_driver()

    try:
        driver.get("https://www.instagram.com/")
        input("인스타 로그인 완료 후 엔터: ")

        post_links = collect_all_post_links(driver, TARGET_ACCOUNT)
        print(f"\n최종 수집 링크 수: {len(post_links)}")

        rows = []

        for i, post_url in enumerate(post_links, 1):
            print(f"\n[{i}/{len(post_links)}] {post_url}")
            try:
                row = crawl_one_post(driver, post_url)
                rows.append(row)
                print(
                    "like_count:", row["like_count"],
                    "| comment_count:", row["comment_count"],
                    "| status:", row["status"]
                )
            except Exception as e:
                print("error:", e)
                rows.append({
                    "post_url":         post_url,
                    "datetime":         None,
                    "caption":          None,
                    "hashtags":         "",
                    "mentions":         "",
                    "like_text":        None,
                    "like_count":       None,
                    "comment_text":     None,
                    "comment_count":    None,
                    "comments_json":    "[]",
                    "comments_texts":   "",
                    "share_count":      None,
                    "expanded_clicks":  0,
                    "meta_description": None,
                    "status":           f"error: {e}",
                })

            pause(1.3, 2.5)
            long_pause()

        df = pd.DataFrame(rows)
        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print(f"\n저장 완료: {OUTPUT_FILE}")
        print(f"총 저장 게시물 수: {len(df)}")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

In [ ]:
# 테스트 영역

In [ ]:
driver_test = create_driver()
driver_test.get("https://www.instagram.com/")
input("로그인 완료 후 엔터: ")  # ← 이거 필수
crawl_one_post(driver_test, 'https://www.instagram.com/hongbeomseok_/p/DT7CZa8EkAW/')